In [19]:
from enum import Enum
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

In [20]:
class XType(Enum):
    STRING = "string"
    DATE = "date"
    NUMERIC = "numeric"


class Trend(Enum):
    INCREASING = "increasing"
    DECREASING = "decreasing"


class Shape(Enum):
    LINEAR = "linear"
    LOGARITHMIC = "logarithmic"
    EXPONENTIAL = "exponential"


@dataclass
class TableConfig:
    """Configuration for a single data table"""
    x_type: XType
    trend: Trend
    shape: Shape
    table_id: int


# Word banks for generating realistic titles and column names
TITLE_ADJECTIVES = [
    "Annual", "Monthly", "Weekly", "Daily", "Quarterly", "Regional", "Global",
    "National", "Local", "Seasonal", "Historical", "Projected", "Estimated",
    "Actual", "Comparative", "Cumulative", "Average", "Total", "Net", "Gross",
    "Primary", "Secondary", "Preliminary", "Final", "Updated", "Revised"
]

TITLE_NOUNS = [
    "Sales", "Revenue", "Expenses", "Profits", "Growth", "Performance",
    "Production", "Output", "Inventory", "Shipments", "Orders", "Customers",
    "Users", "Visitors", "Subscribers", "Members", "Employees", "Staff",
    "Temperature", "Rainfall", "Humidity", "Pressure", "Energy", "Power",
    "Consumption", "Usage", "Traffic", "Volume", "Rate", "Index", "Score",
    "Rating", "Ranking", "Value", "Price", "Cost", "Investment", "Returns",
    "Yield", "Efficiency", "Productivity", "Quality", "Satisfaction", "Engagement"
]

TITLE_CONTEXTS = [
    "Report", "Analysis", "Summary", "Overview", "Breakdown", "Trends",
    "Statistics", "Metrics", "Data", "Figures", "Results", "Measurements"
]

# X column names by type
STRING_X_NAMES = [
    "Category", "Product", "Region", "Department", "Segment", "Group",
    "Division", "Brand", "Channel", "Source", "Type", "Class", "Tier",
    "Level", "Stage", "Phase", "Status", "Mode", "Method", "Approach"
]

DATE_X_NAMES = [
    "Date", "Period", "Month", "Quarter", "Year", "Week", "Day",
    "Timestamp", "Time", "Interval", "Session", "Cycle", "Term"
]

NUMERIC_X_NAMES = [
    "Quantity", "Count", "Number", "Amount", "Units", "Items", "Batch",
    "Sequence", "Index", "Position", "Rank", "Order", "Level", "Grade",
    "Size", "Capacity", "Volume", "Distance", "Duration", "Age"
]

# Y column names (always numeric)
Y_COLUMN_NAMES = [
    "Value", "Amount", "Total", "Sum", "Count", "Quantity", "Score",
    "Rate", "Percentage", "Ratio", "Index", "Measure", "Reading",
    "Output", "Result", "Figure", "Number", "Level", "Metric", "KPI",
    "Revenue", "Cost", "Profit", "Sales", "Units", "Volume", "Growth",
    "Performance", "Efficiency", "Yield", "Return", "Margin", "Average"
]

# String categories for X values
STRING_CATEGORIES = {
    "products": ["Widget A", "Widget B", "Widget C", "Widget D", "Widget E",
                 "Gadget X", "Gadget Y", "Gadget Z", "Device Alpha", "Device Beta"],
    "regions": ["North", "South", "East", "West", "Central",
                "Northeast", "Northwest", "Southeast", "Southwest", "Midwest"],
    "departments": ["Sales", "Marketing", "Engineering", "Finance", "HR",
                    "Operations", "Research", "Support", "Legal", "Admin"],
    "months": ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"][:10],
    "segments": ["Premium", "Standard", "Basic", "Enterprise", "Starter",
                 "Pro", "Elite", "Essential", "Advanced", "Core"],
    "channels": ["Online", "Retail", "Wholesale", "Direct", "Partner",
                 "Mobile", "Desktop", "Social", "Email", "Phone"],
    "brands": ["Alpha", "Beta", "Gamma", "Delta", "Epsilon",
               "Zeta", "Eta", "Theta", "Iota", "Kappa"],
    "tiers": ["Tier 1", "Tier 2", "Tier 3", "Tier 4", "Tier 5",
              "Tier 6", "Tier 7", "Tier 8", "Tier 9", "Tier 10"],
    "grades": ["Grade A", "Grade B", "Grade C", "Grade D", "Grade E",
               "Grade F", "Grade G", "Grade H", "Grade I", "Grade J"],
    "phases": ["Phase 1", "Phase 2", "Phase 3", "Phase 4", "Phase 5",
               "Phase 6", "Phase 7", "Phase 8", "Phase 9", "Phase 10"]
}


def generate_title(x_type: XType, trend: Trend, shape: Shape, used_titles: set) -> str:
    """Generate a unique, realistic title for a data table"""
    max_attempts = 100
    for _ in range(max_attempts):
        adj = random.choice(TITLE_ADJECTIVES)
        noun = random.choice(TITLE_NOUNS)
        context = random.choice(TITLE_CONTEXTS)
        
        # Vary the title format
        formats = [
            f"{adj} {noun} {context}",
            f"{noun} {context}",
            f"{adj} {noun}",
            f"{context} of {adj} {noun}",
            f"{noun} by {random.choice(['Category', 'Period', 'Value'])}",
        ]
        title = random.choice(formats)
        
        if title not in used_titles:
            used_titles.add(title)
            return title
    
    # Fallback with unique ID
    return f"{random.choice(TITLE_ADJECTIVES)} {random.choice(TITLE_NOUNS)} #{random.randint(1000, 9999)}"


def generate_column_names(x_type: XType, used_pairs: set) -> Tuple[str, str]:
    """Generate unique column name pairs based on x_type"""
    max_attempts = 100
    
    x_names = {
        XType.STRING: STRING_X_NAMES,
        XType.DATE: DATE_X_NAMES,
        XType.NUMERIC: NUMERIC_X_NAMES
    }[x_type]
    
    for _ in range(max_attempts):
        x_name = random.choice(x_names)
        y_name = random.choice(Y_COLUMN_NAMES)
        
        # Ensure x and y names are different
        while y_name.lower() == x_name.lower():
            y_name = random.choice(Y_COLUMN_NAMES)
        
        pair = (x_name, y_name)
        if pair not in used_pairs:
            used_pairs.add(pair)
            return pair
    
    # Fallback
    return (random.choice(x_names), random.choice(Y_COLUMN_NAMES))


def generate_x_values(x_type: XType, n_rows: int = 10) -> List[Any]:
    """Generate x values based on type"""
    if x_type == XType.STRING:
        # Randomly select a category type and use its values
        category_type = random.choice(list(STRING_CATEGORIES.keys()))
        values = STRING_CATEGORIES[category_type][:n_rows]
        return values
    
    elif x_type == XType.DATE:
        # Generate sequential dates with random start and interval
        start_year = random.randint(2015, 2024)
        start_month = random.randint(1, 12)
        start_day = random.randint(1, 28)
        start_date = datetime(start_year, start_month, start_day)
        
        # Random interval type
        interval_type = random.choice(["days", "weeks", "months"])
        if interval_type == "days":
            delta = random.randint(1, 7)
            dates = [start_date + timedelta(days=i * delta) for i in range(n_rows)]
        elif interval_type == "weeks":
            dates = [start_date + timedelta(weeks=i) for i in range(n_rows)]
        else:  # months
            dates = []
            for i in range(n_rows):
                month = (start_month - 1 + i) % 12 + 1
                year = start_year + (start_month - 1 + i) // 12
                dates.append(datetime(year, month, min(start_day, 28)))
        
        return [d.strftime("%Y-%m-%d") for d in dates]
    
    else:  # NUMERIC
        # Generate numeric x values with various patterns
        pattern = random.choice(["sequential", "spaced", "random_ordered"])
        
        if pattern == "sequential":
            start = random.randint(1, 100)
            step = random.randint(1, 10)
            values = [start + i * step for i in range(n_rows)]
        elif pattern == "spaced":
            start = random.randint(0, 50)
            spacing = random.uniform(0.5, 5.0)
            values = [round(start + i * spacing, 2) for i in range(n_rows)]
        else:  # random_ordered
            values = sorted(random.sample(range(1, 1000), n_rows))
        
        return values


def generate_y_values(
    n_rows: int,
    trend: Trend,
    shape: Shape,
    noise_level: float = None
) -> List[float]:
    """
    Generate y values based on trend and shape.
    
    Randomizes:
    - Base value range
    - Scale factor
    - Noise level
    - Specific curve parameters
    """
    if noise_level is None:
        noise_level = random.uniform(0.02, 0.15)  # 2-15% noise
    
    # Randomize the base parameters
    base_min = random.uniform(10, 100)
    base_max = random.uniform(base_min * 2, base_min * 10)
    
    # Generate base x for mathematical functions (0 to 1 normalized)
    x = np.linspace(0, 1, n_rows)
    
    if shape == Shape.LINEAR:
        # y = mx + b with random slope
        slope = random.uniform(0.5, 2.0)
        y = x * slope
        
    elif shape == Shape.LOGARITHMIC:
        # y = a * log(bx + 1) with random parameters
        a = random.uniform(0.8, 1.5)
        b = random.uniform(5, 15)
        y = a * np.log(b * x + 1)
        y = y / y.max()  # Normalize to 0-1 range
        
    elif shape == Shape.EXPONENTIAL:
        # y = a * (e^(bx) - 1) with random parameters
        b = random.uniform(1.5, 3.0)
        y = np.exp(b * x) - 1
        y = y / y.max()  # Normalize to 0-1 range
    
    # Scale to desired range
    y = base_min + y * (base_max - base_min)
    
    # Apply trend direction
    if trend == Trend.DECREASING:
        y = y[::-1]
    
    # Add random noise
    noise = np.random.normal(0, noise_level * (base_max - base_min), n_rows)
    y = y + noise
    
    # Ensure positive values and round
    y = np.maximum(y, 0.01)
    y = np.round(y, 2)
    
    return y.tolist()


def generate_single_table(config: TableConfig, used_titles: set, used_column_pairs: set) -> Dict:
    """Generate a single data table with metadata"""
    n_rows = 10
    
    # Generate unique title and column names
    title = generate_title(config.x_type, config.trend, config.shape, used_titles)
    x_col, y_col = generate_column_names(config.x_type, used_column_pairs)
    
    # Generate data
    x_values = generate_x_values(config.x_type, n_rows)
    y_values = generate_y_values(n_rows, config.trend, config.shape)
    
    # Create the table
    table = {
        "id": config.table_id,
        "title": title,
        "metadata": {
            "x_type": config.x_type.value,
            "trend": config.trend.value,
            "shape": config.shape.value
        },
        "columns": {
            "x": x_col,
            "y": y_col
        },
        "data": [
            {x_col: x, y_col: y}
            for x, y in zip(x_values, y_values)
        ]
    }
    
    return table


def generate_all_tables(n_total: int = 3000, seed: int = None) -> List[Dict]:
    """
    Generate all data tables with balanced distribution.
    
    Distribution:
    - 3 x-types × 2 trends × 3 shapes = 18 combinations
    - Each combination gets n_total / 18 tables
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
    
    # Calculate tables per combination
    n_combinations = len(XType) * len(Trend) * len(Shape)  # 18
    base_count = n_total // n_combinations
    remainder = n_total % n_combinations
    
    # Create configuration list with exact distribution
    configs = []
    table_id = 0
    combo_id = 0
    
    for x_type in XType:
        for trend in Trend:
            for shape in Shape:
                # Distribute remainder evenly
                count = base_count + (1 if combo_id < remainder else 0)
                combo_id += 1
                
                for _ in range(count):
                    configs.append(TableConfig(
                        x_type=x_type,
                        trend=trend,
                        shape=shape,
                        table_id=table_id
                    ))
                    table_id += 1
    
    # Shuffle to randomize order (prevent any ordering bias)
    random.shuffle(configs)
    
    # Re-assign IDs after shuffle
    for i, config in enumerate(configs):
        config.table_id = i
    
    # Generate tables
    tables = []
    used_titles = set()
    used_column_pairs = set()
    
    print(f"Generating {n_total} tables...")
    for i, config in enumerate(configs):
        table = generate_single_table(config, used_titles, used_column_pairs)
        tables.append(table)
        
        if (i + 1) % 500 == 0:
            print(f"  Generated {i + 1}/{n_total} tables...")
    
    print(f"Done! Generated {len(tables)} tables.")
    return tables


def verify_distribution(tables: List[Dict]) -> Dict:
    """Verify the distribution of tables across all categories"""
    stats = {
        "total": len(tables),
        "by_x_type": {},
        "by_trend": {},
        "by_shape": {},
        "by_combination": {}
    }
    
    for table in tables:
        meta = table["metadata"]
        x_type = meta["x_type"]
        trend = meta["trend"]
        shape = meta["shape"]
        combo = f"{x_type}_{trend}_{shape}"
        
        stats["by_x_type"][x_type] = stats["by_x_type"].get(x_type, 0) + 1
        stats["by_trend"][trend] = stats["by_trend"].get(trend, 0) + 1
        stats["by_shape"][shape] = stats["by_shape"].get(shape, 0) + 1
        stats["by_combination"][combo] = stats["by_combination"].get(combo, 0) + 1
    
    return stats


def save_tables(tables: List[Dict], output_dir: str = "synthetic_data"):
    """Save tables to files"""
    os.makedirs(output_dir, exist_ok=True)
    
    # Save all tables as single JSON
    all_tables_path = os.path.join(output_dir, "all_tables.json")
    with open(all_tables_path, 'w') as f:
        json.dump(tables, f, indent=2)
    print(f"Saved all tables to: {all_tables_path}")
    
    # Save distribution statistics
    stats = verify_distribution(tables)
    stats_path = os.path.join(output_dir, "distribution_stats.json")
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"Saved distribution stats to: {stats_path}")
    
    return all_tables_path, stats_path


def print_sample_tables(tables: List[Dict], n_samples: int = 3):
    """Print sample tables for verification"""
    print("\n" + "="*70)
    print("SAMPLE TABLES")
    print("="*70)
    
    # Get one sample from each x_type
    for x_type in XType:
        matching = [t for t in tables if t["metadata"]["x_type"] == x_type.value]
        sample = random.choice(matching)
        
        print(f"\n--- {x_type.value.upper()} X-TYPE EXAMPLE ---")
        print(f"Title: {sample['title']}")
        print(f"Metadata: {sample['metadata']}")
        print(f"Columns: {sample['columns']}")
        print("Data:")
        df = pd.DataFrame(sample["data"])
        print(df.to_string(index=False))


def print_distribution_report(stats: Dict):
    """Print a formatted distribution report"""
    print("\n" + "="*70)
    print("DISTRIBUTION REPORT")
    print("="*70)
    
    print(f"\nTotal tables: {stats['total']}")
    
    print("\n--- BY X-TYPE (expecting ~1000 each) ---")
    for x_type, count in sorted(stats["by_x_type"].items()):
        pct = count / stats["total"] * 100
        print(f"  {x_type:12s}: {count:4d} ({pct:.1f}%)")
    
    print("\n--- BY TREND (expecting ~1500 each) ---")
    for trend, count in sorted(stats["by_trend"].items()):
        pct = count / stats["total"] * 100
        print(f"  {trend:12s}: {count:4d} ({pct:.1f}%)")
    
    print("\n--- BY SHAPE (expecting ~1000 each) ---")
    for shape, count in sorted(stats["by_shape"].items()):
        pct = count / stats["total"] * 100
        print(f"  {shape:12s}: {count:4d} ({pct:.1f}%)")
    
    print("\n--- BY COMBINATION (expecting ~166-167 each) ---")
    for combo, count in sorted(stats["by_combination"].items()):
        pct = count / stats["total"] * 100
        print(f"  {combo:35s}: {count:4d} ({pct:.2f}%)")

SEED = 42
# Generate all tables
tables = generate_all_tables(n_total=3000, seed=SEED)

# Verify distribution
stats = verify_distribution(tables)
print_distribution_report(stats)

# Print samples
print_sample_tables(tables, n_samples=3)
# all_tables_path, stats_path = save_tables(tables, output_dir="")

Generating 3000 tables...
  Generated 500/3000 tables...
  Generated 1000/3000 tables...
  Generated 1500/3000 tables...
  Generated 2000/3000 tables...
  Generated 2500/3000 tables...
  Generated 3000/3000 tables...
Done! Generated 3000 tables.

DISTRIBUTION REPORT

Total tables: 3000

--- BY X-TYPE (expecting ~1000 each) ---
  date        : 1002 (33.4%)
  numeric     :  996 (33.2%)
  string      : 1002 (33.4%)

--- BY TREND (expecting ~1500 each) ---
  decreasing  : 1500 (50.0%)
  increasing  : 1500 (50.0%)

--- BY SHAPE (expecting ~1000 each) ---
  exponential : 1000 (33.3%)
  linear      : 1000 (33.3%)
  logarithmic : 1000 (33.3%)

--- BY COMBINATION (expecting ~166-167 each) ---
  date_decreasing_exponential        :  167 (5.57%)
  date_decreasing_linear             :  167 (5.57%)
  date_decreasing_logarithmic        :  167 (5.57%)
  date_increasing_exponential        :  167 (5.57%)
  date_increasing_linear             :  167 (5.57%)
  date_increasing_logarithmic        :  167 (5.